# Seminar HCI and BCI in practice
## Session 1 Introduction to data structure and variables

This seminar is based on data from an experiment in which epilepsy patients with subdurally implanted electrode arrays **(ECoG)** performed different hand gestures to control an avatar through a virtual reality.

 The aim of the original project was to detect gestures from the recorded brain signals. For further information, please read following section ```Experimental paradigm and data structure``` 

 For this seminar we took one of these gestures (wiggling with the index finger) and divided it into two antagonistic parts (flexion and extension).

---

## Experimental paradigm and data structure

Most movement related Brain-Machine-Interface (BMI) research in humans has focused on decoding two or three classes of rather elementary real or imagined movements such as sticking out the tongue or squeezing the hand. However, to be able to control more complex devices with more degrees of freedom one would like to be able to discriminate more movement classes. In humans, gestures of the hand and the arm are increasingly used for computer interfacing. However, little is currently known about the feasibility of using different complex gestures to control BMIs. The aim of this study was to investigate whether subdural ECoG-recordings allow for reliable single trial multi-class discrimination between different gestures as well as different phases within gestures.

**Subjects**

Five patients (S1 – S5) with subdurally placed electrode grids for localization of epileptic foci participated in this study. The location of the grid placement was determined only on medical considerations. Center-to-center electrode spacing was either 1 cm (8x8 electrodes) or 0.4 cm (16x16 electrodes).

This course will focus on the data set from one subject only. The data set from this subject originally includes 256 electrodes with an interelectrode spacing of 0.4 cm. In this course however, we will focus on only 40 of these electrodes, in order to reduce the amount of data.

<img src="./docs/figs/fig1_Electrode_Grid.png" width="1000"/>

**Task**

The patients' task was to use gestures of one hand to navigate through a virtual reality (VR) environment to collect tokens. Gestures were assessed with a data glove equipped with one bend sensor for each finger and a three-axis accelerometer. They performed five different gestures to navigate through the VR-environment: Index finger wiggle, waving the hand up and down with extended fingers, turning the hand palm up and down with extended fingers, turning the fist and waving the fist up and down. Based on the glove sensor data we manually assigned time intervals to gestures.

<img src="./docs/figs/fig2_Experimental.png" width="1000"/>

<img src="./docs/figs/fig3_Gesture_Coding.png" width="1000"/>

**This seminar will only focus on the discrimination of flexion and extension (the two antagonistic phases) of the index finger.**

**Data structure**

The experiment originally resulted in two datasets

1. Neurophysiological recordings (#Channles x #Samples) -> rawEcog.pkl

2. Behavioural data (data glove) -> gloveResamp.pkl

The ‘rawEcog.pkl’ data has already been transformed and all the data relevant for the course is saved in the file ecogStruct.pkl.

---

## Define your own working path

In [ ]:
# Environment Setting
import scipy.io
import numpy as np
import json
import os
import sys
import pickle
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# Self-defined functions
sys.path.append(os.path.join(os.getcwd(), "src"))
from plot_raw_gesture import plot_raw_gesture
from plot_glove_data import plot_glove_data
from plot_own_labels import plot_own_labels

# Define current working directory as your main_path (where you downloaded the seminar scripts and data)
main_path = os.getcwd()
data_path = os.path.join(main_path, 'data','raw')
print(f'Now you are located: {main_path}')

---

## Test glove data set
To make you more familiar with the data you get from a sensor glove we will first look at a test glove data set, that was recorded in order to gain information about the time course of each sensor while performing a particular gesture.

In [ ]:
# Load test data (consisting of the variable gloveResamp)
glove_file = os.path.join(data_path, 'gloveResamp_Dav_2011_9_7_14_28_21.pkl')
with open(glove_file, 'rb') as f:
    gloveResamp = pickle.load(f)

print(gloveResamp.keys())

# now plot the test glove data
plot_raw_gesture(gloveResamp)
# Upper subplot: data from the sensors of the glove (For us the blue trace is the sensor of interest, as it indicates index finger wiggling)
# Lower subplot: Accelerometer values (Cartesian coordinates)

---

<h2 style="color: #FF0000; font-weight: bold;">TASK 1 (1 Point):</h2>

Try to find the interval in which the subject did the finger wiggling gesture. There are four other gestures (See `Experimental paradigm and data structure`). Can you discriminate them? Write down the gestures you can identify and their approximate time.

<h3 style="color: #FF0000; font-weight: bold;">Your Answers or Code demostration:</h3>

...

---

## Analyzing the glove data of our subject

Now continue with the data from our subject:

In [ ]:
# First load the following data files to your workspace: 

with open(os.path.join(data_path, 'gloveResamp.pkl'), 'rb') as f:
    gloveResamp = pickle.load(f)  # Motion data from the sensor glove

with open(os.path.join(data_path, 'ecogAnalog.pkl'), 'rb') as f:
    ecogAnalog = pickle.load(f)   # contains data from an analog channel to synchronize brain and glove data 
# (it is important to synchronize when using the glove data to find the gesture onsets, that are later used to analyze the brain data)

with open(os.path.join(data_path, 'epoch.pkl'), 'rb') as f:
    epoch = pickle.load(f)        # provides onsets and labels of gestures (hand labeled) 

The gesture onsets and labels contained in the variable epoch were found by hand labeling. 

In *Task 2* you will try this hand labeling yourself.
**BEFORE** you start with the task, plot the hand labeled data saved in epoch, so you know how it will look like when finished (We will plot the glove data time series with onsets of flexion and extension of the index finger.)

In [ ]:
plot_glove_data(gloveResamp,ecogAnalog,epoch)

**Notes:**  As you can see it is hard to recognize anything in this figure. It is best to zoom in using the **zoom tool (magnifying glass)**. Click on the zoom-in symbol (magnifying glass) and choose a time window in your figure. If you want to scroll to other time windows, click on the **coordinate system symbol (scrolling)**

The vertical lines represent the onsets of the epochs (**normal blue lines = flexion epochs; dashed blue lines = extension epochs**) The epochs will be **0.25 seconds** long. If the time of flexion or extension is longer than 0.5 seconds than two epochs will be fitted within one gesture. For better understanding of how the onsets are calculated, have a look at the `getEpochs` function.

Now close this figure again and try finding your own epochs.

<h2 style="color: #FF0000; font-weight: bold;">TASK 2:</h2>

Remember the following steps are just to give you an idea of the procedure used to cut and label the data. Afterwards we will continue using the onsets and labels already saved in epoch.

In order to cut and label the data the function ``` getEpochs.py ``` is used. This function first asks you for graphical input, that you will provide by mouse clicks on the figure that will appear. To make own epochs **select the beginning, the reversal point and the end of one gesture cycle (flexion + extension) with the mouse** (crosshairs will appear to make this easier). **Mark always three points otherwise data are rejected**. From this input the function then calculates flexion and extension onsets. After finishing one gesture, you will be asked if you want to continue with a next gesture, type either ```'y'(es)``` to continue or ```'n'(o)``` to end process. **IMPORTANT** is to optimize your figure size (zoom-in like described above) before starting with the first gesture. 

*Try it out for at least five gestures.*

In [ ]:
# Before marking your own epochs, save the necessary data for later access in the terminal.
# (Run in Terminal, as the Notebook kernel runs extremely slow for this task)
with open(os.path.join(data_path, "data_for_own_epoch.pkl"), "wb") as f:
    pickle.dump({"gloveResamp": gloveResamp, "ecogAnalog": ecogAnalog}, f)

print("✅ Data successfully saved to `data_for_own_epoch.pkl` in main_path, ready for terminal access.")

<div 
    style="border: 1px dashed black; border-radius: 10px; padding: 10px;">
    
## HOWTO: Run `get_epochs.py`

### Arguments:
| Argument | Short | Default                             | Description |
| :--- | :--- |:------------------------------------| :--- |
| `--input` | `-i` | `./data/raw/data_for_own_epoch.pkl` | Path to the input data file. |
| `--output` | `-o` | `./results/epochs.pkl`              | Path where the labels will be saved. |
| `--epoch-length`| `-l` | `0.25`                              | Target length of each epoch in seconds. |
| `--keep-markers`| `-k` | `False`                             | If set, markers from previous clicks remain on screen. |

### Usage:
1. Open your terminal
2. Navigate to the directory of the folder 'BCI_2026'
3. Switch to the Anaconda environment where you have installed the necessary dependencies (e.g., `conda activate BCI_Seminar_2026`)
4. Run the following command:
```bash
python get_epochs.py --input ./data/raw//data_for_own_epoch.pkl --output .results/epochs.pkl --epoch-length 0.25 --keep-markers False
```

or simply:
```bash
python get_epochs.py
```
</div>

In [ ]:
#Code lines for running get_epochs.py(Copy the output and run in a new anaconda terminal)
print('cd', main_path)
print('python get_epochs.py')

<h3 style="color: #FF0000; font-weight: bold;">TASK 2.1 (1 Point)</h3>

Find the onsets of the 5 gestures you defined. Copy them in here:


<h3 style="color: #FF0000; font-weight: bold;">Your Answers or Code demostration:</h3>

...

In [ ]:
# After label your own epochs, now load the labeled epochs
with open(r".\results\epochs.pkl", "rb") as f:
    ownEpochs = pickle.load(f)

# check data
print("loaded own labeled epoch data:", ownEpochs)

# Let's have a look of your own labeled epochs
plot_own_labels(gloveResamp,ecogAnalog,ownEpochs)

---

<h2 style="color: #FF0000; font-weight: bold;">TASK 3 (Discussion, 1 Point):</h2>

Check if only the index finger moved in the periods you labeled. Think about the appropriateness of the label if more than one finger moved. 


<h3 style="color: #FF0000; font-weight: bold;">Your Answers or Code demostration:</h3>

...


---

## Anatomy
Now have a short look at the anatomy of our subject. Raw ECoG data were recorded from a 16x16 electrode grid with a center-to-center spacing of 0.4 cm. To reduce the amount of data, only 40 of the 256 electrodes available, will be used in this course.

Plot anatomical image of brain and electrodes

<h2 style="color: #FF0000; font-weight: bold;">TASK 4 （1 Point):</h2>

Figure out how to read in and plot the anatomical data in Jupyter Notebook

`Anatomy = './docs/figs/GP33_anatomy_40_electrodes.png' `

<h3 style="color: #FF0000; font-weight: bold;">Fill in the missing parts (...) in the code below</h3>

In [ ]:
# Read and show anatomical picture
Anatomy = './docs/figs/GP33_anatomy_40_electrodes.png'
img_path = os.path.join(main_path, Anatomy)

img = mpimg....

# plot show
plt.figure(figsize=(20, 20))
plt.imshow(img)
plt.axis("off")  # axis off
plt.show()
# parameter for range

---

## ECoG data
The data is saved in a `dict` called `ecog`. Most of the functions we use in this seminar come from our ecog toolbox and are based on this kind of data structure.

In [ ]:
with open(os.path.join(data_path, 'ecogStruct.pkl'), 'rb') as f:
    ecog = pickle.load(f) 

# now have a look at the current state of the ecog structure
ecog.keys()

In [ ]:
# Go through the keys of the dictionary 'ecog' and check the class and dimensions of data
for k in ecog.keys():
    print(f'{k}, class: {type(ecog[k]).__name__}, shape: {np.array(ecog[k]).shape}')
# Note: if NumPy returns shape=(), that means 0-dim data, which also means it is a scalar

<h2 style="color: #FF0000; font-weight: bold;">TASK 5 （1 Point):</h2>

Try to understand what all the fields in the ecog structure contain. 

- What are the meanings of each key-value pair in the `dict` `ecog`?
- How is the shape of data under each key?
- How is the data saved in the `dict` `ecog` (data-type/ class)?


<h3 style="color: #FF0000; font-weight: bold;">Your Answers or Code demostration:</h3>

...

---

## Preprocessing of ECoG Data

The first preprocessing step is baseline correction. Here, the mean across the samples within one channel is subtracted from each sample in the respective channel, setting the average of each channel to zero.

<h2 style="color: #FF0000; font-weight: bold;">TASK 6 (Discussion, 1 Point)</h2>

What is the goal of the baseline correction here? Can you think of other ways of achieving this goal?

<h3 style="color: #FF0000; font-weight: bold;">Your Answers or Code demostration:</h3>

...


<h2 style="color: #FF0000; font-weight: bold;">TASK 7 (2 pt)</h2>

Implement two ways of performing baseline correction.
1. Using a for-loop over channels. (If you want to get some challenge, try list comperhension instead of a for-loop)
2. Creating a vector of channel means and subtracting them from all samples simultaneously.

<h3 style="color: #FF0000; font-weight: bold;">Fill in the missing parts (...) in the code below</h3>

In [ ]:
# Baseline Correction
ecog_data = np.array(ecog['data'])

# Method 1:
ecog_bc_1  = ...

# Method 2:
ecog_bc_2 = ...

# Extracting a sample channel
before_correction = ecog['data'][0][100000:110000]  # Original data
after_correction = ecog_bc_1[0][100000:110000]
time_points = np.array(ecog['timebase'])[100000:110000]

# Create the figure with two subplots
fig, axs = plt.subplots(2, 1, figsize=(10, 6))

# First subplot: Before baseline correction
axs[0].plot(time_points, before_correction)
axs[0].set_title('Before baseline correction')
axs[0].set_xlabel('Time')
axs[0].set_ylabel('Amplitude')

# Second subplot: After baseline correction
axs[1].plot(time_points, after_correction)
axs[1].set_title('After baseline correction')
axs[1].set_xlabel('Time')
axs[1].set_ylabel('Amplitude')

# Adjust layout and show the plot
plt.tight_layout()
plt.show()